# Step 3: Course-Level Prerequisite Mining

Mining prerequisite relationships between courses based on:
1. Temporal precedence
2. Success conditioning (pass/fail)

Rule: Course A → Course B if P(pass B | pass A) > threshold

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import json

spark = SparkSession.builder \
    .appName("Step3_CoursePrerequisites") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Thresholds
MIN_SUCCESS_RATE = 0.4  # P(pass B | pass A) >= 0.65
MIN_SUPPORT = 20  # At least 15 users
W_TEMPORAL = 0.6
W_SUCCESS = 0.4

print(f"✓ Spark {spark.version} ready")

✓ Spark 4.1.1 ready


## Step 1: Load Course Sequences

In [5]:
# Load user sequences
user_seq = spark.read.json(f"{OUTPUT_PATH}/user_sequences_course_json")

print(f"Loaded {user_seq.count():,} users")

# Explode sequences
courses_df = user_seq.select(
    "user_id",
    explode("sequence").alias("course")
).select(
    "user_id",
    col("course.course_id").alias("course_id"),
    col("course.sequence_order").alias("order"),
    col("course.R_score").alias("R_score"),
    col("course.passed").alias("passed")
)

courses_df.cache()
print(f"Total courses: {courses_df.count():,}")
courses_df.show(10)

Loaded 11,385 users
Total courses: 71,479
+--------+---------+-----+-------+------+
| user_id|course_id|order|R_score|passed|
+--------+---------+-----+-------+------+
|U_101079| C_696954|    1|  0.307|     0|
|U_101079| C_682458|    2|  0.307|     0|
|U_101079| C_770784|    3|  0.307|     0|
|U_101079| C_854839|    4|  0.307|     0|
| U_11731| C_629514|    7|  0.307|     0|
| U_11731| C_654554|    8|  0.307|     0|
| U_11731| C_655850|    9|  0.307|     0|
| U_11731| C_629503|   10|  0.307|     0|
| U_11731| C_655852|   13|  0.307|     0|
| U_11731| C_597314|   14|  0.307|     0|
+--------+---------+-----+-------+------+
only showing top 10 rows


## Step 2: Create Course Pairs

In [9]:
# Self-join để tạo temporal pairs
c1 = courses_df.select(
    col("user_id"),
    col("course_id").alias("course_a"),
    col("order").alias("order_a"),
    col("passed").alias("passed_a"),
    col("R_score").alias("R_score_a")
)

c2 = courses_df.select(
    col("user_id"),
    col("course_id").alias("course_b"),
    col("order").alias("order_b"),
    col("passed").alias("passed_b"),
    col("R_score").alias("R_score_b")
)

# Join: course_a before course_b
pairs = c1.join(
    c2,
    (c1.user_id == c2.user_id) & (c1.order_a < c2.order_b)
).select(
    c1.user_id,
    c1.course_a,
    c2.course_b,
    c1.passed_a,
    c2.passed_b,
    c1.R_score_a,
    c2.R_score_b
)

pairs.cache()
print(f"Course pairs: {pairs.count():,}")
pairs.show(10)

Course pairs: 714,160
+--------+--------+--------+--------+--------+---------+---------+
| user_id|course_a|course_b|passed_a|passed_b|R_score_a|R_score_b|
+--------+--------+--------+--------+--------+---------+---------+
|U_101079|C_696954|C_854839|       0|       0|    0.307|    0.307|
|U_101079|C_696954|C_770784|       0|       0|    0.307|    0.307|
|U_101079|C_696954|C_682458|       0|       0|    0.307|    0.307|
|U_101079|C_682458|C_854839|       0|       0|    0.307|    0.307|
|U_101079|C_682458|C_770784|       0|       0|    0.307|    0.307|
|U_101079|C_770784|C_854839|       0|       0|    0.307|    0.307|
| U_11731|C_629514|C_682792|       0|       0|    0.307|    0.307|
| U_11731|C_629514|C_681460|       0|       0|    0.307|    0.307|
| U_11731|C_629514|C_697151|       0|       0|    0.307|    0.307|
| U_11731|C_629514|C_682716|       0|       0|    0.307|    0.307|
+--------+--------+--------+--------+--------+---------+---------+
only showing top 10 rows


## Step 3: Calculate Temporal Support

In [20]:
# Count temporal occurrences
temporal_counts = pairs.groupBy("course_a", "course_b").agg(
    count("*").alias("temporal_count"),
    count_distinct("user_id").alias("unique_users")
)

print(f"Temporal pairs: {temporal_counts.count():,}")
temporal_counts.show(10)

Temporal pairs: 246,332
+---------+---------+--------------+------------+
| course_a| course_b|temporal_count|unique_users|
+---------+---------+--------------+------------+
| C_682240| C_696597|             4|           4|
| C_696972| C_680987|             3|           3|
| C_680931| C_677030|             2|           2|
| C_697034| C_681678|             2|           2|
| C_697032| C_734015|             3|           3|
| C_735244|C_1755907|             1|           1|
|C_1755921| C_676917|             1|           1|
| C_681001| C_696911|             2|           2|
| C_681671| C_707089|             1|           1|
| C_676642| C_697091|             6|           6|
+---------+---------+--------------+------------+
only showing top 10 rows


## Step 4: Calculate Success Conditioning

In [22]:
# Filter mastery: passed_a = 1
mastery = pairs.filter(col("passed_a") == 1)

mastery_counts = mastery.groupBy("course_a", "course_b").agg(
    count("*").alias("mastery_count")
)

# Filter success: passed_a = 1 AND passed_b = 1
success = pairs.filter((col("passed_a") == 1) & (col("passed_b") == 1))

success_counts = success.groupBy("course_a", "course_b").agg(
    count("*").alias("success_count")
)

print(f"Success pairs: {success_counts.count():,}")

Success pairs: 168,380


## Step 5: Calculate Prerequisite Strength

In [23]:
# Join all counts
prereq = temporal_counts.join(
    mastery_counts, ["course_a", "course_b"], "left"
).join(
    success_counts, ["course_a", "course_b"], "left"
).fillna(0)

# Calculate probabilities
prereq = prereq.withColumn(
    "P_temporal",
    lit(1.0)  # Always 1.0 for course pairs (already temporal)
).withColumn(
    "P_success",
    when(col("mastery_count") > 0, col("success_count") / col("mastery_count")).otherwise(0.0)
).withColumn(
    "strength",
    W_TEMPORAL * col("P_temporal") + W_SUCCESS * col("P_success")
)

# Filter
prereq_filtered = prereq.filter(
    (col("P_success") >= MIN_SUCCESS_RATE) &
    (col("unique_users") >= MIN_SUPPORT)
)

final_count = prereq_filtered.count()
print(f"Final prerequisites: {final_count:,}")
prereq_filtered.show(20)

Final prerequisites: 3,258
+---------+--------+--------------+------------+-------------+-------------+----------+---------+--------+
| course_a|course_b|temporal_count|unique_users|mastery_count|success_count|P_temporal|P_success|strength|
+---------+--------+--------------+------------+-------------+-------------+----------+---------+--------+
|C_1410156|C_883345|            26|          26|           11|           11|       1.0|      1.0|     1.0|
| C_629559|C_676932|           107|         107|           41|           41|       1.0|      1.0|     1.0|
| C_629559|C_677010|            40|          40|           18|           18|       1.0|      1.0|     1.0|
| C_629559|C_680745|            29|          29|           13|           13|       1.0|      1.0|     1.0|
| C_629559|C_696942|            57|          57|           17|           17|       1.0|      1.0|     1.0|
| C_629559|C_697374|            21|          21|            7|            7|       1.0|      1.0|     1.0|
| C_629559

## Step 6: Analysis

In [24]:
# Stats
print("Prerequisite Statistics:")
prereq_filtered.select(
    count("*").alias("count"),
    avg("strength").alias("avg_strength"),
    max("strength").alias("max_strength"),
    avg("P_success").alias("avg_success_rate"),
    avg("unique_users").alias("avg_support")
).show()

# Top prerequisites
print("\nTop 20 Prerequisites:")
prereq_filtered.orderBy(col("strength").desc()).show(20)

Prerequisite Statistics:
+-----+------------+------------+----------------+------------------+
|count|avg_strength|max_strength|avg_success_rate|       avg_support|
+-----+------------+------------+----------------+------------------+
| 3258|         1.0|         1.0|             1.0|39.983118477593614|
+-----+------------+------------+----------------+------------------+


Top 20 Prerequisites:
+---------+--------+--------------+------------+-------------+-------------+----------+---------+--------+
| course_a|course_b|temporal_count|unique_users|mastery_count|success_count|P_temporal|P_success|strength|
+---------+--------+--------------+------------+-------------+-------------+----------+---------+--------+
| C_629559|C_677076|            21|          21|            8|            8|       1.0|      1.0|     1.0|
| C_629559|C_697032|            79|          79|           16|           16|       1.0|      1.0|     1.0|
| C_629559|C_694136|            27|          27|            6|    

## Step 7: Export

In [26]:
# Export
prereq_filtered.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/course_prerequisites")

# JSON
rows = prereq_filtered.collect()
prereq_dict = {}
for row in rows:
    a = row["course_a"]
    if a not in prereq_dict:
        prereq_dict[a] = {}
    prereq_dict[a][row["course_b"]] = {
        "strength": row["strength"],
        "P_success": row["P_success"],
        "support": row["unique_users"],
        "success_count": row["success_count"],
        "mastery_count": row["mastery_count"]
    }

import pickle
with open(f"{OUTPUT_PATH}/course_prerequisites.pkl", "wb") as f:
    pickle.dump(prereq_dict, f)

with open(f"{OUTPUT_PATH}/course_prerequisites.json", "w", encoding='utf-8') as f:
    json.dump(prereq_dict, f, indent=2)

print(f"Saved {len(prereq_dict)} courses with prerequisites")
# Cleanup
pairs.unpersist()
courses_df.unpersist()

print("\n" + "="*50)
print("STEP 3 COMPLETE!")
print("="*50)

Saved 179 courses with prerequisites

STEP 3 COMPLETE!
